# Undercurrent quickstart

*See the undercurrent before it surfaces.*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wrynx/undercurrent/blob/main/notebooks/quickstart.ipynb)

Undercurrent is Wrynx's activation-probing platform. It attaches small **probes** to a language
model's internal activations while it generates, so you can score, flag or stop a generation based
on what the model is computing, not only on the text it has already written. This notebook runs
GPT-2 on a CPU (or a free Colab runtime) in a few minutes and covers the core loop: write a spec,
attach a probe, read per-request results, abort a generation from inside the model, and log
observations to a file without touching the response.

## 1. Install

The base install includes the Hugging Face backend, so no extras are needed.

In [ ]:
%pip install -q undercurrent
# Or the latest development version from GitHub `main`:
# %pip install -q "undercurrent @ git+https://github.com/wrynx/undercurrent.git@main"

## 2. Pick a model and write a spec

A **spec** lists *extraction points*: which tensor to capture (`tensor_type`), at which layers,
at which token positions, and which probe receives it. Here we capture the residual stream at a
middle layer for every generated token (`generated[*]`).

`MODEL` defaults to `openai-community/gpt2`; set the `UNDERCURRENT_NB_MODEL` environment variable to try another
model (the automated test uses a tiny random GPT-2).

In [ ]:
import os

from transformers import AutoConfig

MODEL = os.environ.get("UNDERCURRENT_NB_MODEL", "openai-community/gpt2")
LAYER = AutoConfig.from_pretrained(MODEL).num_hidden_layers // 2  # 6 for gpt2
print(f"model={MODEL} layer={LAYER}")

SPEC = f"""
version: "1"
extraction_points:
  - name: resid_norm
    layers: {LAYER}
    tensor_type: residual_stream
    position: "generated[*]"
    probe_type: resid_norm
    probe_kind: single_shot
"""

## 3. A function probe

The simplest probe is a plain function `(record) -> float | bool | ProbeSignal | None`, decorated
with `@probe`. It receives one `ActivationRecord` per matching activation (`record.tensor` is the
captured tensor) and returns a score. Scores at or above `threshold` are flagged; `action="flag"`
records the flag and lets generation continue (the default, `"abort"`, would stop it).

`on_result=` is a callback that receives every finished generation.

In [ ]:
from undercurrent.core import probe
from undercurrent.model import ProbedModel


@probe("resid_norm", threshold=100.0, action="flag")
def resid_norm(record) -> float:
    return float(record.tensor.float().norm())


def show(out):
    verdict = out.probe_results["resid_norm"].verdict
    print(f"{out.prompt!r} -> flagged={verdict['flagged']} max_norm={verdict['max_score']:.1f} tokens={verdict['n']}")


with ProbedModel.from_pretrained(MODEL, spec=SPEC, on_result=show) as model:
    outputs = model.generate(
        ["The weather today is", "My favourite programming language is"],
        max_new_tokens=20,
        temperature=0.0,
    )

Each output carries the generated text and one `ProbeResult` per extraction point. A function
probe's verdict summarises the request; `signal_history` keeps the per-token scores.

In [ ]:
out = outputs[0]
print(repr(out.text))
print([round(s.metadata["score"], 1) for s in out.probe_results["resid_norm"].signal_history])

## 4. A trajectory probe that aborts

Probes that keep state across tokens are classes with `probe_kind = "trajectory"`. This one is a
small **repetition detector**: it compares each generated token's residual stream with every
earlier one and aborts once the model has been retracing its own steps (cosine similarity at or
above `threshold`) for `patience` tokens in a row. It emits `FLAG` signals while the streak
builds and one `ABORT` when it completes.

It runs **inline** (the default `execution_mode`), so its `ABORT` signal stops generation
mid-stream. The output reports `aborted=True`, which extraction point stopped it, and the text
generated up to that point. `probe_args` in the spec are passed to the probe's constructor.

In [ ]:
import torch

from undercurrent.core import Probe, ProbeAction, ProbeResult, ProbeSignal


class RepetitionProbe(Probe):
    probe_kind = "trajectory"

    def __init__(self, threshold: float = 0.98, patience: int = 3) -> None:
        super().__init__()
        self.threshold = threshold
        self.patience = patience

    def on_start(self, ctx):
        self.seen = []  # one residual vector per generated token
        self.streak = 0
        self.history = []

    def on_activation(self, record):
        x = record.tensor.float().flatten()
        sim = max((float(torch.cosine_similarity(x, h, dim=0)) for h in self.seen), default=0.0)
        self.seen.append(x)
        self.streak = self.streak + 1 if sim >= self.threshold else 0
        if self.streak == 0 or self.streak > self.patience:
            return None  # nothing new to report (or already aborted)
        action = ProbeAction.ABORT if self.streak == self.patience else ProbeAction.FLAG
        signal = ProbeSignal(
            action=action, confidence=sim, metadata={"similarity": round(sim, 4), "pos": record.token_pos}
        )
        self.history.append(signal)
        return signal

    def on_end(self, ctx):
        return ProbeResult(
            ctx.request_id,
            self.extraction_point_name,
            verdict={"tokens": len(self.seen), "repeating": self.streak >= self.patience},
            signal_history=self.history,
        )


LOOP_SPEC = f"""
version: "1"
extraction_points:
  - name: loop_guard
    layers: {LAYER}
    tensor_type: residual_stream
    position: "generated[*]"
    probe_type: repetition
    probe_kind: trajectory
    probe_args: {{threshold: 0.98, patience: 3}}
"""

with ProbedModel.from_pretrained(MODEL, spec=LOOP_SPEC, probes={"repetition": RepetitionProbe}) as model:
    for prompt in ["The weather today is", "I love you. I love you. I love you."]:
        out = model.generate(prompt, max_new_tokens=40, temperature=0.0)
        print(f"prompt:  {prompt!r}")
        print(f"aborted: {out.aborted}  ({out.abort_reason})")
        print(f"text:    {out.text!r}")
        print()

## 5. Observe mode: log, don't intervene

Set `execution_mode: async` and the same probe runs **off the generation path**, on a worker
pool: generation never waits for it, and its signals are observations rather than interventions.
They go to a log sink. `FileLogSink` appends one JSON object per line (NDJSON) for every signal
and every final result. Production deployments can use `WebhookLogSink` (retries,
dead-lettering, redaction) instead.

In [ ]:
from pathlib import Path

from undercurrent.sinks import FileLogSink

LOG = Path("observations.ndjson")
LOG.unlink(missing_ok=True)

OBSERVE_SPEC = LOOP_SPEC.replace("probe_kind: trajectory", "probe_kind: trajectory\n    execution_mode: async")

with ProbedModel.from_pretrained(
    MODEL, spec=OBSERVE_SPEC, probes={"repetition": RepetitionProbe}, log_sink=FileLogSink(LOG)
) as model:
    out = model.generate("I love you. I love you. I love you.", max_new_tokens=40, temperature=0.0)

print(f"aborted: {out.aborted}  (observe mode never stops generation)")
print(f"text:    {out.text!r}")

Read the log back. Each line has `kind` (`signal` or `result`), `request_id`,
`extraction_point_name`, `timestamp` and a `payload`.

In [ ]:
import json

records = [json.loads(line) for line in LOG.read_text().splitlines()]

print(f"{'kind':<8} {'point':<12} {'action':<8} details")
for r in records:
    p = r["payload"]
    if r["kind"] == "signal":
        print(f"{r['kind']:<8} {r['extraction_point_name']:<12} {p['action']:<8} {p['metadata']}")
    else:
        print(f"{r['kind']:<8} {r['extraction_point_name']:<12} {'':<8} verdict={p['verdict']}")

## 6. A fuller demo: content safety

The repository includes an end-to-end **demo** in
[`examples/content_safety/`](https://github.com/wrynx/undercurrent/tree/main/examples/content_safety): MLP probes over a
Llama-style model, YAML specs, a vLLM pipeline script and a Dockerfile. It is an example, not part
of the installed package, so clone the repository to run it.

## Next steps

- **Documentation**: [https://wrynx.github.io/undercurrent/](https://wrynx.github.io/undercurrent/), including the
  [quickstart](https://wrynx.github.io/undercurrent/getting-started/quickstart/),
  [probes and their lifecycle](https://wrynx.github.io/undercurrent/concepts/probes/),
  [inline vs async execution](https://wrynx.github.io/undercurrent/concepts/execution-modes/),
  [writing a custom probe](https://wrynx.github.io/undercurrent/guides/custom-probe/) and
  [observation sinks](https://wrynx.github.io/undercurrent/guides/observation-sinks/).
- **Train your own probe**: the [training notebook](https://github.com/wrynx/undercurrent/blob/main/notebooks/train_probe.ipynb)
  collects activations, fits a probe and plugs it into a spec.
- **Source, issues and discussions**: [github.com/wrynx/undercurrent](https://github.com/wrynx/undercurrent).